<a href="https://colab.research.google.com/github/Pruthvi226/Minor_Project/blob/main/00_setup_and_data_access.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Setup & Data Access — Session 1

**Cost-, Fidelity-, and Shift-Aware Active Learning for Computational Pathology**

This is `00_setup_and_data_access.ipynb`, built from **SESSION 1 — START HERE** in the
fact-checked v2 kickoff brief. Run every cell top to bottom before writing any modeling code.

What this notebook does:

1. Checks what compute you actually have this session.
2. Mounts Drive and builds the project folder structure.
3. Files/checks the access requests everything downstream depends on (HuggingFace gated
   weights, Kaggle/PANDA, MIDOG sign-up).
4. Runs the CIFAR-10/100 pipeline sanity check (zero access dependencies).
5. Confirms the CAMELYON S3 mirror and TCGA/GDC bucket are reachable.
6. Writes shared `src/paths.py` and `src/utils.py` so no later notebook hardcodes a path or
   reinvents seeding.
7. Creates `DECISIONS.md`, `README.md`, and `requirements.txt`.

If any step fails — gated access still pending, a bucket unreachable, quota exhausted —
record that plainly in `DECISIONS.md`. Don't skip ahead, and don't fabricate a "looks fine"
result for a step that didn't actually run.

## 1. Check your GPU

Free tier gives you a T4 most of the time, occasionally nothing during high-demand windows,
occasionally a v5e TPU instead. Know what you actually have before planning compute budget.

In [ ]:
!nvidia-smi


/bin/bash: line 1: nvidia-smi: command not found


## 2. Mount Drive & build the project structure

Data, embeddings, checkpoints, and results all live on Drive so a 90-minute idle disconnect
doesn't cost you anything already saved.

In [ ]:
from google.colab import drive

drive.mount('/content/drive')


Mounted at /content/drive


In [ ]:
import os

PROJECT_ROOT = "/content/drive/MyDrive/al_pathology"

FOLDERS = [
    "data/camelyon",
    "data/panda",
    "data/midog",
    "data/tcga",
    "data/cptac",
    "embeddings/uni2h",
    "embeddings/conch",
    "notebooks",
    "src",
    "checkpoints",
    "results/figures",
    "results/tables",
    "logs",
]

for folder in FOLDERS:
    path = os.path.join(PROJECT_ROOT, folder)
    os.makedirs(path, exist_ok=True)
    print(f"OK  {path}")

print("\nProject root:", PROJECT_ROOT)


OK  /content/drive/MyDrive/al_pathology/data/camelyon
OK  /content/drive/MyDrive/al_pathology/data/panda
OK  /content/drive/MyDrive/al_pathology/data/midog
OK  /content/drive/MyDrive/al_pathology/data/tcga
OK  /content/drive/MyDrive/al_pathology/data/cptac
OK  /content/drive/MyDrive/al_pathology/embeddings/uni2h
OK  /content/drive/MyDrive/al_pathology/embeddings/conch
OK  /content/drive/MyDrive/al_pathology/notebooks
OK  /content/drive/MyDrive/al_pathology/src
OK  /content/drive/MyDrive/al_pathology/checkpoints
OK  /content/drive/MyDrive/al_pathology/results/figures
OK  /content/drive/MyDrive/al_pathology/results/tables
OK  /content/drive/MyDrive/al_pathology/logs

Project root: /content/drive/MyDrive/al_pathology


## 3. File every access request now — none are instant

- **HuggingFace (gated, needs institutional email as your HF primary email):**
  - UNI2-h — https://huggingface.co/MahmoodLab/UNI2-h
  - CONCH — https://huggingface.co/MahmoodLab/CONCH
    (the *code* is open and pip-installable; the *weights* need this same gated approval)
- **Kaggle:** create an account, generate an API token at kaggle.com/settings, and accept
  the competition rules at
  https://www.kaggle.com/c/prostate-cancer-grade-assessment/rules
- **MIDOG:** sign up and verify at https://midog2025.grand-challenge.org (or
  https://midog.deepmicroscopy.org) — sign-up only, no API, so there's no cell for this step.

Run the two cells below to start the HuggingFace and Kaggle steps from inside Colab.

In [ ]:
# HuggingFace login. This only authenticates your token — it does NOT grant access to a
# gated repo by itself. You still need to click "Agree and access repository" on each of the
# UNI2-h and CONCH pages above, with your institutional email set as your HF primary email.
!pip install -q huggingface_hub
from huggingface_hub import notebook_login

notebook_login()


In [ ]:
# Check whether gated access has actually cleared yet for each repo.
from huggingface_hub import HfApi

api = HfApi()
for repo_id in ["MahmoodLab/UNI2-h", "MahmoodLab/CONCH"]:
    try:
        api.model_info(repo_id)
        print(f"[ACCESS OK]      {repo_id}")
    except Exception as e:
        print(f"[NOT YET / ERR]  {repo_id}: {type(e).__name__}")


[ACCESS OK]      MahmoodLab/UNI2-h
[ACCESS OK]      MahmoodLab/CONCH


In [ ]:
import os
from getpass import getpass

# Prompt user for Kaggle username and API key
kaggle_username = input("Enter your Kaggle username: ")
kaggle_key = getpass("Enter your Kaggle API key: ")

# Set environment variables
os.environ['KAGGLE_USERNAME'] = kaggle_username
os.environ['KAGGLE_KEY'] = kaggle_key

# Create .kaggle directory if it doesn't exist
os.makedirs("/root/.kaggle", exist_ok=True)

# Create kaggle.json file
kaggle_json_content = f'{{"username":"{kaggle_username}","key":"{kaggle_key}"}}'
with open("/root/.kaggle/kaggle.json", "w") as f:
    f.write(kaggle_json_content)

# Set permissions for kaggle.json
os.chmod("/root/.kaggle/kaggle.json", 0o600)

print("Kaggle credentials configured successfully.")

Enter your Kaggle username: pruthvirajpanduga
Enter your Kaggle API key: ··········
Kaggle credentials configured successfully.


In [ ]:
# Verify the credential works AND that you've accepted the PANDA competition rules —
# this call fails clearly if either is missing.
!pip install -q kaggle
!kaggle competitions files -c prostate-cancer-grade-assessment


401 Client Error: Unauthorized for url: https://api.kaggle.com/v1/competitions.CompetitionApiService/ListDataFiles


## 4. CIFAR-10 / CIFAR-100 pipeline sanity check

Zero access dependencies. This just proves Colab, torchvision, Drive mounting, and disk I/O
all work end to end before anything gated or gigapixel enters the picture.

In [ ]:
import torchvision

print("Downloading CIFAR-10...")
cifar10 = torchvision.datasets.CIFAR10(root="/content/data", train=True, download=True)
print("Downloading CIFAR-100...")
cifar100 = torchvision.datasets.CIFAR100(root="/content/data", train=True, download=True)

print(f"\nCIFAR-10:  {len(cifar10)} training images")
print(f"CIFAR-100: {len(cifar100)} training images")
print("Sanity check passed.")


100%|██████████| 170M/170M [23:10<00:00, 123kB/s]


100%|██████████| 169M/169M [17:25<00:00, 162kB/s]



CIFAR-10:  50000 training images
CIFAR-100: 50000 training images
Sanity check passed.


## 5. Confirm the CAMELYON S3 mirror is reachable

AWS Open Data Sponsorship Program mirror, no AWS account or credentials needed. This only
lists the bucket — do **not** run a full `aws s3 sync` in this session, the corpus is many
TB. (Reference for later: `aws s3 sync s3://camelyon-dataset/ /content/drive/MyDrive/data/camelyon --no-sign-request`,
and use the FULL release rather than the WILDS `camelyon17` wrapper — the WILDS version
collapses everything to patch-level tumor/normal labels and drops the lesion-level XML and
patient-level pN-stage distinction this project's cost axis depends on.)

In [ ]:
!pip install -q awscli
!aws s3 ls --no-sign-request s3://camelyon-dataset/


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.2/20.2 MB 27.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.5/570.5 kB 27.0 MB/s eta 0:00:00
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
sphinx 8.2.3 requires docutils<0.22,>=0.20, but you have docutils 0.19 which is incompatible.
                           PRE CAMELYON16/
                           PRE CAMELYON17/


## 6. Confirm the TCGA / GDC bucket is reachable

Open bucket, no special auth normally required. If this prompts for credentials instead of
listing, run `!gcloud auth login` once and retry.

In [ ]:
!gsutil ls gs://gdc-tcga-phs000178-open/ | head -20


BucketNotFoundException: 404 gs://gdc-tcga-phs000178-open bucket does not exist.


## 7. Shared `src/paths.py` and `src/utils.py`

Every later notebook imports from these instead of hardcoding a Drive path or reinventing
seeding — required for the "no hardcoded absolute paths" and "minimum 3 seeds per
configuration" items in the project's definition of done.

In [ ]:
paths_content = '''# Shared path constants for the AL pathology project.
# Import this everywhere instead of hardcoding "/content/drive/MyDrive/..." paths.
# If you ever move the project, change PROJECT_ROOT here once.

import os

PROJECT_ROOT = "/content/drive/MyDrive/al_pathology"

DATA_DIR = os.path.join(PROJECT_ROOT, "data")
CAMELYON_DIR = os.path.join(DATA_DIR, "camelyon")
PANDA_DIR = os.path.join(DATA_DIR, "panda")
MIDOG_DIR = os.path.join(DATA_DIR, "midog")
TCGA_DIR = os.path.join(DATA_DIR, "tcga")
CPTAC_DIR = os.path.join(DATA_DIR, "cptac")

EMBEDDINGS_DIR = os.path.join(PROJECT_ROOT, "embeddings")
UNI2H_EMB_DIR = os.path.join(EMBEDDINGS_DIR, "uni2h")
CONCH_EMB_DIR = os.path.join(EMBEDDINGS_DIR, "conch")

NOTEBOOKS_DIR = os.path.join(PROJECT_ROOT, "notebooks")
SRC_DIR = os.path.join(PROJECT_ROOT, "src")
CHECKPOINTS_DIR = os.path.join(PROJECT_ROOT, "checkpoints")

RESULTS_DIR = os.path.join(PROJECT_ROOT, "results")
FIGURES_DIR = os.path.join(RESULTS_DIR, "figures")
TABLES_DIR = os.path.join(RESULTS_DIR, "tables")

LOGS_DIR = os.path.join(PROJECT_ROOT, "logs")
'''

paths_file = os.path.join(PROJECT_ROOT, "src", "paths.py")
with open(paths_file, "w") as f:
    f.write(paths_content)
print(f"Wrote {paths_file}")


Wrote /content/drive/MyDrive/al_pathology/src/paths.py


In [ ]:
utils_content = '''# Shared seeding utility for the AL pathology project.
# Call set_seed(s) at the start of every experiment, and log which seed you used.

import random

import numpy as np
import torch


def set_seed(seed: int) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


# Minimum required by the project quality bar: 3 seeds per configuration.
SEEDS = [0, 1, 2]
'''

utils_file = os.path.join(PROJECT_ROOT, "src", "utils.py")
with open(utils_file, "w") as f:
    f.write(utils_content)
print(f"Wrote {utils_file}")


Wrote /content/drive/MyDrive/al_pathology/src/utils.py


## 8. `DECISIONS.md`, `README.md`, `requirements.txt`

Created once here, then appended to for the rest of the project — every reproduce-before-you-
trust check, every subsetting decision, every correction, goes in `DECISIONS.md`.

In [ ]:
from datetime import date

today = date.today().isoformat()

decisions_content = f'''# DECISIONS.md

Project: Cost-, Fidelity-, and Shift-Aware Active Learning for Computational Pathology

This log starts from the fact-checked v2 kickoff brief. Two errors from v1 were corrected
before this project began: the BPAL publication date (2024 to Oct 2025) and the claim that
CONCH's pretrained weights are ungated (they are gated on HuggingFace, same as UNI2-h). Two
v1 citations are flagged [UNVERIFIED] and must be independently re-checked before the
related-work section cites them: the S-DOTA-style stain and scanner augmentation 2023
citation, and the 2024 generative-replay continual learning citation.

## Session 1 - {today}

- GPU this session: FILL IN from the nvidia-smi output above
- Access requests filed today:
  - [ ] HuggingFace UNI2-h (MahmoodLab/UNI2-h) - status: pending / approved
  - [ ] HuggingFace CONCH (MahmoodLab/CONCH) - status: pending / approved
  - [ ] Kaggle account created, PANDA competition rules accepted - status:
  - [ ] grand-challenge.org account, MIDOG 2025 registration - status:
- CIFAR-10 / CIFAR-100 sanity check: pass / fail
- CAMELYON S3 mirror reachability check: pass / fail (paste a one-line summary of the ls output)
- TCGA / GDC bucket reachability check: pass / fail
- Sampling procedure to use if any dataset needs subsetting for compute reasons: FILL IN NOW,
  before you need it. Per the project quality bar, subsetting must be random or stratified
  and logged here before use, never chosen after seeing results.

## Template for future entries

### [Date] - [Decision title]

Context:

Decision:

Alternatives considered:

Why this way:
'''

decisions_path = os.path.join(PROJECT_ROOT, "DECISIONS.md")
with open(decisions_path, "w") as f:
    f.write(decisions_content)
print(f"Wrote {decisions_path}")


Wrote /content/drive/MyDrive/al_pathology/DECISIONS.md


In [ ]:
readme_content = '''# Cost-, Fidelity-, and Shift-Aware Active Learning for Computational Pathology

A learned, foundation-model-conditioned active learning framework that decides not only
which whole-slide image or region to annotate next, but at what label granularity - slide
level, region of interest, or full pixel-level segmentation - subject to a real annotation
time budget and an empirically estimated scanner/center shift signal.

Started from a fact-checked v2 kickoff brief (September 2026). See DECISIONS.md for the
running log of what was checked, what was corrected, and every experimental decision made
along the way.

## Notebook structure

Run in order. Each notebook imports shared code from src/ rather than redefining paths or
seeds locally.

00_setup_and_data_access.ipynb                  - this notebook: installs, Drive mount, access checks
01_foundation_model_embedding_extraction.ipynb  - obtain/verify UNI2-h and CONCH embeddings
02_baselines.ipynb                              - generic AL baselines on CIFAR (sanity) and one real WSI benchmark
03_cost_model_annotation_granularity.ipynb      - calibrate the multi-tier cost model
04_utility_predictor.ipynb                      - the core novel component, unit-tested in isolation
05_al_simulation_loop.ipynb                     - the retrain/relabel loop, budgeted by real cost
06a_experiments_camelyon17_metastasis.ipynb
06b_experiments_panda_grading.ipynb
06c_experiments_midog_mitosis.ipynb
06d_experiments_tcga_cptac_joint.ipynb
07_ablations.ipynb
08_analysis_and_figures.ipynb                   - every evaluation figure, with seed variance shown

## Reproducing this project

1. Open these notebooks in Colab, or mount this Drive folder directly.
2. Run 00_setup_and_data_access.ipynb first, every session - it mounts Drive, checks the GPU,
   and verifies every external access path before anything else runs.
3. Install pinned dependencies from requirements.txt.
4. Every experiment sets its seed explicitly via src/utils.py:set_seed() and uses the
   minimum 3-seed protocol in src/utils.py:SEEDS. Report mean plus/minus std in every table
   and figure, never a single-seed number.
5. All paths come from src/paths.py. Nothing downstream hardcodes an absolute path.

## Quality bar (full detail in the kickoff brief)

- No fabricated results: if a cell cannot run, that is logged in DECISIONS.md, not filled in
- No dataset modification: only partitioning by metadata the dataset already ships with
- Every baseline reproduces a published number (CIFAR-10 mechanics check, plus one real WSI
  benchmark) before any new number is trusted
- Minimum 3 seeds per configuration, mean and spread reported everywhere
- DECISIONS.md is complete enough that someone else could reconstruct the Methods section
'''

readme_path = os.path.join(PROJECT_ROOT, "README.md")
with open(readme_path, "w") as f:
    f.write(readme_content)
print(f"Wrote {readme_path}")


Wrote /content/drive/MyDrive/al_pathology/README.md


In [ ]:
requirements_content = '''torch
torchvision
timm
transformers
huggingface_hub
kaggle
awscli
google-cloud-storage
openslide-python
scikit-learn
scikit-image
pandas
numpy
matplotlib
seaborn
tqdm
pyyaml
opencv-python-headless
'''

requirements_path = os.path.join(PROJECT_ROOT, "requirements.txt")
with open(requirements_path, "w") as f:
    f.write(requirements_content)
print(f"Wrote {requirements_path}")


Wrote /content/drive/MyDrive/al_pathology/requirements.txt


## Session 1 checklist

- [ ] GPU checked
- [ ] Drive mounted, folder structure created
- [ ] UNI2-h access requested
- [ ] CONCH access requested
- [ ] Kaggle account + PANDA rules accepted
- [ ] MIDOG 2025 sign-up submitted
- [ ] CIFAR-10 / CIFAR-100 sanity check passed
- [ ] CAMELYON S3 mirror confirmed reachable
- [ ] TCGA/GDC bucket confirmed reachable
- [ ] `src/paths.py` and `src/utils.py` written
- [ ] `DECISIONS.md`, `README.md`, `requirements.txt` written and filled in (not left as
      placeholders)

**Next:** once HuggingFace access clears, move to
`01_foundation_model_embedding_extraction.ipynb` and pull the precomputed UNI2-h embeddings
for TCGA, CPTAC, and PANDA first — the single biggest compute-savings move available, since
it skips gigapixel-image handling entirely for two of your four datasets.